# 04 — INT8 Fault Injection

YOLOv8s | Single-bit weight fault injection | Unit tests and boundary tests

This notebook documents preliminary INT8 weight-level tests. It does not perform full INT8 model inference or COCO evaluation.

In [ ]:
%pip install -q ultralytics==8.4.189

In [ ]:
import ultralytics

print("Ultralytics:", ultralytics.__version__)
assert ultralytics.__version__ == "8.4.189"

print("ULTRALYTICS INSTALL: PASS")

In [ ]:
from pathlib import Path
import os

mount_path = Path("/content/drive")
project_path = mount_path / "MyDrive/YOLOv8_Fault_Tolerance"

print("Mount path exists:", mount_path.exists())
print("Mount path is symlink:", mount_path.is_symlink())

if mount_path.is_dir():
    print("Mount path contents:", os.listdir(mount_path)[:10])

print("Project path exists:", project_path.exists())
print("DRIVE PATH INSPECTION: PASS")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

PROJECT_DIR = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance"
)

assert PROJECT_DIR.is_dir(), "Project directory not accessible"

MODEL_PATH = Path("/content/yolov8s.pt")

if not MODEL_PATH.is_file():
    candidates = list(PROJECT_DIR.rglob("yolov8s.pt"))
    if len(candidates) == 1:
        MODEL_PATH = candidates[0]
    else:
        print("yolov8s.pt not found locally. Downloading official YOLOv8s weights...")
        # This will download yolov8s.pt to the current working directory
        from ultralytics.utils.downloads import download
        download('https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s.pt', dir=MODEL_PATH.parent)
        MODEL_PATH = Path("/content/yolov8s.pt")

model = YOLO(str(MODEL_PATH)).model
model.eval()
model.cpu()

conv_layers = {
    name: layer
    for name, layer in model.named_modules()
    if isinstance(layer, torch.nn.Conv2d)
}

assert len(conv_layers) == 64
assert all(
    layer.weight.dtype == torch.float32
    for layer in conv_layers.values()
)

print("Model path:", MODEL_PATH)
print("Conv2d layers:", len(conv_layers))
print("Model device:", next(model.parameters()).device)
print("FAULT INJECTION SETUP: PASS")

In [ ]:
import hashlib

EXPECTED_SHA256 = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

actual_sha256 = sha256_file(MODEL_PATH)

print("Model:", MODEL_PATH)
print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256:  ", actual_sha256)

assert actual_sha256 == EXPECTED_SHA256, (
    "MODEL HASH MISMATCH: Stop before fault injection."
)

print("\nMODEL REPRODUCIBILITY CHECK: PASS")

In [ ]:
import torch

# Select the first convolution layer
layer_name = "model.0.conv"
layer = conv_layers[layer_name]

# Work on a separate CPU copy
weights = layer.weight.detach().cpu().clone()
original_weights = weights.clone()

# Symmetric per-tensor INT8 quantization
scale = weights.abs().max().item() / 127.0
assert scale > 0

q_weights = torch.clamp(
    torch.round(weights / scale),
    -127, 127
).to(torch.int8)

# Choose one weight deterministically
weight_index = (0, 0, 0, 0)
original_q = int(q_weights[weight_index].item())

# Flip bit 0 (least significant bit)
bit_position = 0
before_bits = original_q & 0xFF
after_bits = before_bits ^ (1 << bit_position)

# Convert back to signed INT8
faulty_q = after_bits if after_bits < 128 else after_bits - 256

# Fault is applied ONLY to a copy
faulty_q_weights = q_weights.clone()
faulty_q_weights[weight_index] = faulty_q

# Dequantize the selected values
clean_value = original_q * scale
faulty_value = faulty_q * scale

# Verification
assert (before_bits ^ after_bits).bit_count() == 1
assert int(q_weights[weight_index]) == original_q
assert int(faulty_q_weights[weight_index]) == faulty_q
assert torch.equal(weights, original_weights)
assert torch.equal(
    layer.weight.detach().cpu(),
    original_weights
)

print("Layer:", layer_name)
print("Weight index:", weight_index)
print("Bit position:", bit_position)
print("INT8 before:", format(before_bits, "08b"))
print("INT8 after: ", format(after_bits, "08b"))
print("Quantized value before:", original_q)
print("Quantized value after:", faulty_q)
print("Dequantized value before:", clean_value)
print("Dequantized value after:", faulty_value)
print("Absolute difference:", abs(faulty_value - clean_value))
print("Original model unchanged: True")

print("\nINT8 BIT-FLIP UNIT TEST: PASS")

In [ ]:
def flip_int8_bit(q_tensor, index, bit_position):
    """
    Flip exactly one bit in a signed INT8 tensor copy.
    Original tensor remains unchanged.
    """
    assert q_tensor.dtype == torch.int8
    assert 0 <= bit_position <= 7

    faulty = q_tensor.clone()

    before = int(q_tensor[index].item())
    before_bits = before & 0xFF

    after_bits = before_bits ^ (1 << bit_position)
    after = after_bits if after_bits < 128 else after_bits - 256

    faulty[index] = after

    assert (before_bits ^ after_bits).bit_count() == 1
    assert int(q_tensor[index].item()) == before

    record = {
        "index": tuple(index),
        "bit_position": bit_position,
        "value_before": before,
        "value_after": after,
        "bits_before": format(before_bits, "08b"),
        "bits_after": format(after_bits, "08b"),
    }

    return faulty, record


# Verify the reusable function
test_faulty, test_record = flip_int8_bit(
    q_weights,
    index=(0, 0, 0, 0),
    bit_position=0
)

assert test_record["value_before"] == 4
assert test_record["value_after"] == 5
assert torch.equal(test_faulty, faulty_q_weights)
assert torch.equal(
    layer.weight.detach().cpu(),
    original_weights
)

print(test_record)
print("\nREUSABLE INT8 INJECTOR: PASS")

In [ ]:
# Test all 8 bit positions on the same quantized weight
test_index = (0, 0, 0, 0)
original_tensor = q_weights.clone()

results = []

for bit in range(8):
    faulty_tensor, record = flip_int8_bit(
        q_weights,
        index=test_index,
        bit_position=bit
    )

    # Exactly one tensor element must change
    changed_elements = int(
        torch.count_nonzero(faulty_tensor != q_weights).item()
    )

    assert changed_elements == 1
    assert record["value_before"] == 4
    assert (
        int(record["bits_before"], 2)
        ^ int(record["bits_after"], 2)
    ) == (1 << bit)

    results.append(record)

# Ensure original quantized weights remain unchanged
assert torch.equal(q_weights, original_tensor)

# Ensure original FP32 model remains unchanged
assert torch.equal(
    layer.weight.detach().cpu(),
    original_weights
)

for r in results:
    print(
        f"Bit {r['bit_position']}: "
        f"{r['bits_before']} -> {r['bits_after']} | "
        f"{r['value_before']} -> {r['value_after']}"
    )

print("\nALL 8 INT8 BITS TEST: PASS")

In [ ]:
import torch

boundary_values = [-128, -127, -1, 0, 1, 126, 127]
test_tensor = torch.tensor(boundary_values, dtype=torch.int8)
original_tensor = test_tensor.clone()

test_count = 0

for i, value in enumerate(boundary_values):
    for bit in range(8):
        faulty, record = flip_int8_bit(
            test_tensor,
            index=(i,),
            bit_position=bit
        )

        expected_bits = (value & 0xFF) ^ (1 << bit)
        expected_value = (
            expected_bits - 256
            if expected_bits >= 128
            else expected_bits
        )

        assert record["value_before"] == value
        assert record["value_after"] == expected_value
        assert int(faulty[i]) == expected_value

        # Only the selected element changes
        assert int(torch.count_nonzero(faulty != test_tensor)) == 1

        test_count += 1

assert torch.equal(test_tensor, original_tensor)

print("Boundary values:", boundary_values)
print("Tests passed:", test_count)
print("Original tensor unchanged:", True)
print("\nINT8 BOUNDARY TESTS: PASS")

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
SRC_DIR = PROJECT / "code_recovered" / "src"

assert (SRC_DIR / "int8_fault_injection.py").is_file(), (
    "INT8 fault injection module not found"
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from int8_fault_injection import flip_int8_bit

q = torch.tensor([4], dtype=torch.int8)
faulty, record = flip_int8_bit(q, (0,), 7)

assert q.item() == 4
assert faulty.item() == -124
assert record["bit_position"] == 7

print("REUSABLE MODULE CHECK: PASS")